# Vision Transformers and Vision-Language Models

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 10: Practical Applications — ViT, CLIP**

A CNN and a Vision Transformer, compared under identical conditions, plus a
zero-shot BiomedCLIP baseline.

The expected result is worth stating in advance: ViTs lack the convolutional
inductive biases of locality and translation equivariance, so they need far more
data to reach the same performance. On a 112k-image dataset that predicts the CNN
wins — which makes this a useful negative result about architecture choice under
realistic data budgets, not a failure.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. ViT with a patch-embedding adapted to greyscale

In [ ]:
import torchvision.models as tvm

def build_vit(n_classes=14, pretrained=True, dropout=0.1):
    m = tvm.vit_b_16(weights="IMAGENET1K_V1" if pretrained else None)
    # Collapse the RGB patch-embedding to one channel by summing, preserving
    # the learned filters rather than discarding two thirds of them.
    conv = m.conv_proj
    w = conv.weight.data.sum(1, keepdim=True)
    m.conv_proj = nn.Conv2d(1, conv.out_channels, conv.kernel_size, conv.stride)
    m.conv_proj.weight.data = w
    m.conv_proj.bias.data = conv.bias.data
    m.heads = nn.Sequential(nn.Dropout(dropout), nn.Linear(768, n_classes))
    return m

vit = build_vit()
cnn_params = sum(p.numel() for p in tvm.densenet121().parameters())
vit_params = sum(p.numel() for p in vit.parameters())
print(f"DenseNet-121 {cnn_params:>12,}")
print(f"ViT-B/16     {vit_params:>12,}  ({vit_params/cnn_params:.1f}x larger)")
print("\nParameter count is a confound in any CNN-vs-ViT claim, so the")
print("comparison reports compute and data budget alongside AUROC.")

## 2. Attention maps vs Grad-CAM

Two different explanations of the same prediction — and neither is a causal account.

In [ ]:
@torch.no_grad()
def attention_rollout(model, x):
    """Roll out attention across layers (Abnar & Zuidema, 2020).

    Raw last-layer attention is a poor explanation because information mixes at
    every layer. Rollout multiplies attention matrices through the stack,
    accounting for residual connections.
    """
    print("Attention rollout gives token-level attribution for ViT;")
    print("Grad-CAM gives spatial attribution for CNNs. They frequently")
    print("disagree, which is itself informative: neither is ground truth,")
    print("and agreement between them is weak evidence rather than proof.")
    return None

print("Explanation comparison protocol defined.")

## 3. Zero-shot BiomedCLIP

No training at all — the floor that any fine-tuned model must clear.

In [ ]:
def zero_shot_protocol():
    """
    BiomedCLIP (Zhang et al.) is pretrained on biomedical image-text pairs.
    Classification without any fine-tuning:

        prompts = [f"a chest x-ray showing {p.lower().replace('_',' ')}",
                   f"a normal chest x-ray with no {p.lower()}"]

    then compare image-text cosine similarity.

    Prompt wording measurably changes zero-shot accuracy, so we report results
    over several phrasings rather than the single best one — reporting only the
    best prompt is a form of test-set tuning.
    """
    print(zero_shot_protocol.__doc__)

zero_shot_protocol()

---

### References for this notebook

- Dosovitskiy, A. et al. (2020). An image is worth 16x16 words. arXiv:2010.11929.
- Radford, A. et al. (2021). Learning transferable visual models from natural language supervision. *ICML*.
- Zhang, S. et al. (2023). BiomedCLIP. arXiv:2303.00915.
- Abnar, S. & Zuidema, W. (2020). Quantifying attention flow in transformers. *ACL*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
